# Nemotron SFT Training (Kaggle / RTX PRO 6000) — smoke

Runs the convergent public 0.85 SFT recipe + `completion_only_loss=True`. Current configuration: `sample_n=200`, `max_steps=50`, `max_length=2048`, `packing=False`, `use_qlora=True`. Edit `scripts/build_kaggle_notebooks.py` and re-run with different flags to change.

Data sources required: this code dataset (`aleixlopez/nemotron-code`), the base model (`metric/nemotron-3-nano-30b-a3b-bf16/transformers/default`), and the kienngx CoT corpus (`kienngx/nemotron-30b-competition-trainingdata-cot-labels`).


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Glob-discover instead of hardcoding — Kaggle's path conventions vary
# (`/kaggle/input/datasets/<owner>/<slug>/` for our newer dataset,
#  `/kaggle/input/<slug>/` for older public ones).
import glob, sys
matches = glob.glob("/kaggle/input/**/blackwell_setup.py", recursive=True)
assert matches, "code dataset not attached (no blackwell_setup.py found under /kaggle/input)"
CODE_ROOT = os.path.dirname(matches[0])
print(f"CODE_ROOT = {CODE_ROOT}")
sys.path.insert(0, CODE_ROOT)

import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Pre-model-load Blackwell setup. The mamba_ssm stub MUST be installed before
# AutoModelForCausalLM.from_pretrained loads modeling_nemotron_h.py.
from blackwell_setup import (  # noqa: E402
    set_cuda_alloc_env,
    set_blackwell_ptxas_env,
    install_mamba_ssm_stub,
)
set_cuda_alloc_env()
set_blackwell_ptxas_env()
install_mamba_ssm_stub()
print("CUDA alloc env + Blackwell ptxas env + mamba_ssm stub ready.")


In [ ]:
# Training-only deps installed from the public offline-packages dataset
# (`dennisfong/nvidia-nemotron-offline-packages`). Kaggle's base image already
# ships torch + transformers + datasets; we add trl/peft/accelerate for SFT.
# Internet must be OFF for competition compliance, so `--no-index --find-links`
# is mandatory.

import glob, os, subprocess, sys

# Walk /kaggle/input one level deep to surface what's actually attached.
print("== /kaggle/input layout ==")
for d in sorted(os.listdir("/kaggle/input")):
    sub = f"/kaggle/input/{d}"
    print(f"  {d}/")
    try:
        for f in sorted(os.listdir(sub))[:6]:
            print(f"    {f}/" if os.path.isdir(f"{sub}/{f}") else f"    {f}")
    except Exception as e:
        print(f"    <err: {e}>")

# Discover offline-packages dir by searching for a known-present wheel.
matches = glob.glob("/kaggle/input/**/offline_packages", recursive=True)
if not matches:
    # Some datasets mount the wheels at the dataset root, not a subdir.
    wheels = glob.glob("/kaggle/input/**/trl-*.whl", recursive=True)
    assert wheels, "no offline_packages dir AND no trl-*.whl wheel found"
    OFFLINE_DIR = os.path.dirname(wheels[0])
else:
    OFFLINE_DIR = matches[0]
print(f"\nOFFLINE_DIR = {OFFLINE_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir",
    "--no-index", "--find-links", OFFLINE_DIR,
    "trl", "peft", "accelerate", "bitsandbytes",
])
print("trl + peft + accelerate + bitsandbytes installed from offline packages.")


In [ ]:
from pathlib import Path

# Use the kernel-metadata.json-attached mount directly (faster than kagglehub.download).
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
assert BASE_MODEL_PATH.is_dir(), f"base model not mounted at {BASE_MODEL_PATH}"
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")
print("  files:", sorted(p.name for p in BASE_MODEL_PATH.iterdir())[:6], "...")

# val_split_600.csv ships inside the code dataset (see scripts/sync_code_to_kaggle.py).
VAL_SPLIT = Path(CODE_ROOT) / "data" / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Locate the kienngx CoT corpus dataset. Attach it to this kernel before running:
#   Dataset slug: kienngx/nemotron-30b-competition-trainingdata-cot-labels
from pathlib import Path
import glob
import shutil

CORPUS_CANDIDATES = glob.glob("/kaggle/input/**/final_Nemotron_training_data.csv", recursive=True)
assert CORPUS_CANDIDATES, "Attach the kienngx CoT corpus dataset before running."
CORPUS_PATH = Path(CORPUS_CANDIDATES[0])
print(f"CORPUS_PATH: {CORPUS_PATH}")

OUTPUT_DIR = Path("/kaggle/working/adapter")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)

# Free stale run artifacts to avoid papermill "no space left on device".
for stale in ("/kaggle/working/submission.zip", "/kaggle/working/debug_predictions.csv"):
    p = Path(stale)
    if p.exists():
        p.unlink()

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working free: {free / 1024**3:.2f} GB")


In [ ]:
# smoke run.
from nemotron_train import train

train(
    data_csv=str(CORPUS_PATH),
    output_dir=str(OUTPUT_DIR),
    base_model=str(BASE_MODEL_PATH),
    sample_n=200,        # rows to subsample from the corpus
    max_steps=50,      # cap on optimizer steps (-1 = epochs-driven)
    max_length=2048,    # sequence length cap
    packing=False,          # packing=False (required: eager attn incompatible with packing)
    use_qlora=True,      # QLoRA: load base in NF4 4-bit to fit on 96 GB Blackwell
    extra_training_kwargs={"save_strategy": "no"},  # avoid checkpoint bloat on Kaggle disk
)


In [ ]:
# Package the adapter as submission.zip in /kaggle/working (the Kaggle "Submit" button
# zips notebook outputs; this matches the host's expected zip layout).
import zipfile, os
from pathlib import Path

ADAPTER_DIR = OUTPUT_DIR  # trainer may save here or under OUTPUT_DIR/final
ZIP_PATH = Path("/kaggle/working/submission.zip")

REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]
present = set(os.listdir(ADAPTER_DIR)) if ADAPTER_DIR.exists() else set()
missing = [f for f in REQUIRED if f not in present]
if missing and (ADAPTER_DIR / "final").is_dir():
    ADAPTER_DIR = ADAPTER_DIR / "final"
    present = set(os.listdir(ADAPTER_DIR))
    missing = [f for f in REQUIRED if f not in present]
assert not missing, f"missing required adapter files in {ADAPTER_DIR}: {missing}"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(ADAPTER_DIR / f, arcname=f)
print(f"wrote {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
